In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Relative path - works on any machine
base_path = Path.cwd().parent / 'data' / 'raw'

# Load all datasets
students = pd.read_csv(base_path / 'studentInfo.csv')
assessments = pd.read_csv(base_path / 'assessments.csv')
student_assessments = pd.read_csv(base_path / 'studentAssessment.csv')
student_registration = pd.read_csv(base_path / 'studentRegistration.csv')
student_vle = pd.read_csv(base_path / 'studentVle.csv')
courses = pd.read_csv(base_path / 'courses.csv')
vle = pd.read_csv(base_path / 'vle.csv')

print("All files loaded successfully!")

All files loaded successfully!


In [2]:
# Fill missing imd_band with most common value
students['imd_band'] = students['imd_band'].fillna(
    students['imd_band'].mode()[0]
)

# Verify no more missing values
print("Missing values in students table:")
print(students.isnull().sum())

Missing values in students table:
code_module             0
code_presentation       0
id_student              0
gender                  0
region                  0
highest_education       0
imd_band                0
age_band                0
num_of_prev_attempts    0
studied_credits         0
disability              0
final_result            0
dtype: int64


In [3]:
# Encode categorical variables
students['gender_encoded'] = (students['gender'] == 'M').astype(int)
students['disability_encoded'] = (students['disability'] == 'Y').astype(int)

# Encode age band
age_mapping = {
    '0-35': 0,
    '35-55': 1,
    '55<=': 2
}
students['age_encoded'] = students['age_band'].map(age_mapping)

# Encode education level
education_mapping = {
    'No Formal quals': 0,
    'Lower Than A Level': 1,
    'A Level or Equivalent': 2,
    'HE Qualification': 3,
    'Post Graduate Qualification': 4
}
students['education_encoded'] = students['highest_education'].map(education_mapping)

print("Demographic features created!")
print(students[['gender_encoded', 'disability_encoded', 
                'age_encoded', 'education_encoded']].head())

Demographic features created!
   gender_encoded  disability_encoded  age_encoded  education_encoded
0               1                   0            2                  3
1               0                   0            1                  3
2               0                   1            1                  2
3               0                   0            1                  2
4               0                   0            0                  1


In [4]:
# Total clicks per student
total_clicks = student_vle.groupby('id_student')['sum_click'].agg([
    ('total_clicks', 'sum'),
    ('avg_daily_clicks', 'mean'),
    ('active_days', 'count'),
    ('max_clicks_day', 'max')
]).reset_index()

# Early engagement - clicks in first 30 days
early_vle = student_vle[student_vle['date'] <= 30]
early_clicks = early_vle.groupby('id_student')['sum_click'].sum().reset_index()
early_clicks.columns = ['id_student', 'early_clicks']

print("VLE features created!")
print(total_clicks.head())

VLE features created!
   id_student  total_clicks  avg_daily_clicks  active_days  max_clicks_day
0        6516          2791          4.216012          662              49
1        8462           656          2.157895          304              16
2       11391           934          4.765306          196              76
3       23629           161          2.728814           59              13
4       23698           910          2.983607          305              78


In [5]:
# Fill missing scores with 0 (didn't submit = 0)
student_assessments['score'] = student_assessments['score'].fillna(0)

# Average score per student
assessment_features = student_assessments.groupby('id_student')['score'].agg([
    ('avg_score', 'mean'),
    ('max_score', 'max'),
    ('min_score', 'min'),
    ('num_assessments', 'count')
]).reset_index()

# Late submissions - powerful predictor
student_assessments_merged = student_assessments.merge(
    assessments[['id_assessment', 'date']], 
    on='id_assessment', 
    how='left'
)
student_assessments_merged['is_late'] = (
    student_assessments_merged['date_submitted'] > 
    student_assessments_merged['date']
).astype(int)

late_features = student_assessments_merged.groupby('id_student')['is_late'].agg([
    ('num_late_submissions', 'sum'),
    ('late_submission_rate', 'mean')
]).reset_index()

assessment_features = assessment_features.merge(late_features, on='id_student', how='left')

print("Assessment features created!")
print(assessment_features.head())

Assessment features created!
   id_student  avg_score  max_score  min_score  num_assessments  \
0        6516  61.800000       77.0       48.0                5   
1        8462  87.000000       93.0       83.0                7   
2       11391  82.000000       85.0       78.0                5   
3       23629  82.500000      100.0       63.0                4   
4       23698  74.444444       94.0       56.0                9   

   num_late_submissions  late_submission_rate  
0                     0              0.000000  
1                     1              0.142857  
2                     0              0.000000  
3                     3              0.750000  
4                     4              0.444444  


In [6]:
# Fill missing unregistration date - means student stayed enrolled
student_registration['stayed_enrolled'] = (
    student_registration['date_unregistration'].isnull()
).astype(int)

# Days before course start that student registered
student_registration['date_registration'] = student_registration[
    'date_registration'].fillna(0)

registration_features = student_registration.groupby('id_student').agg(
    stayed_enrolled=('stayed_enrolled', 'max'),
    avg_registration_date=('date_registration', 'mean')
).reset_index()

print("Registration features created!")
print(registration_features.head())

Registration features created!
   id_student  stayed_enrolled  avg_registration_date
0        3733                0                  -68.0
1        6516                1                  -52.0
2        8462                0                  -87.5
3       11391                1                 -159.0
4       23629                1                  -47.0


In [7]:
# Start with students as base
final_df = students.copy()

# Merge VLE features
final_df = final_df.merge(total_clicks, on='id_student', how='left')
final_df = final_df.merge(early_clicks, on='id_student', how='left')

# Merge assessment features
final_df = final_df.merge(assessment_features, on='id_student', how='left')

# Merge registration features
final_df = final_df.merge(registration_features, on='id_student', how='left')

# Fill any remaining nulls with 0
final_df = final_df.fillna(0)

print("Final dataset shape:", final_df.shape)
print("\nAll columns:")
print(final_df.columns.tolist())

Final dataset shape: (32593, 29)

All columns:
['code_module', 'code_presentation', 'id_student', 'gender', 'region', 'highest_education', 'imd_band', 'age_band', 'num_of_prev_attempts', 'studied_credits', 'disability', 'final_result', 'gender_encoded', 'disability_encoded', 'age_encoded', 'education_encoded', 'total_clicks', 'avg_daily_clicks', 'active_days', 'max_clicks_day', 'early_clicks', 'avg_score', 'max_score', 'min_score', 'num_assessments', 'num_late_submissions', 'late_submission_rate', 'stayed_enrolled', 'avg_registration_date']


In [8]:
# Save to processed folder
processed_path = Path.cwd().parent / 'data' / 'processed'
final_df.to_csv(processed_path / 'features.csv', index=False)

print("Features saved successfully!")
print(f"Final shape: {final_df.shape}")
print(f"\nFeature summary:")
print(final_df.describe())

Features saved successfully!
Final shape: (32593, 29)

Feature summary:
         id_student  num_of_prev_attempts  studied_credits  gender_encoded  \
count  3.259300e+04          32593.000000     32593.000000    32593.000000   
mean   7.066877e+05              0.163225        79.758691        0.548431   
std    5.491673e+05              0.479758        41.071900        0.497657   
min    3.733000e+03              0.000000        30.000000        0.000000   
25%    5.085730e+05              0.000000        60.000000        0.000000   
50%    5.903100e+05              0.000000        60.000000        1.000000   
75%    6.444530e+05              0.000000       120.000000        1.000000   
max    2.716795e+06              6.000000       655.000000        1.000000   

       disability_encoded   age_encoded  education_encoded  total_clicks  \
count        32593.000000  32593.000000       32593.000000  32593.000000   
mean             0.097076      0.302672           1.739331   1479.033412 